Created with the help of chat.ai using model Anthropic Claude Sonnet 5

In [1]:
import sqlite3
import re
import pandas as pd
from molmass import Formula

In [2]:
def parse_msp_stream(filepath):
    meta, peaks, in_peaks = {}, [], False

    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip() == "":
                if meta:
                    yield {"meta": meta, "peaks": peaks}
                meta, peaks, in_peaks = {}, [], False
                continue

            if re.match(r"^Num\s*Peaks", line, re.IGNORECASE):
                in_peaks = True
                continue

            if in_peaks:
                parts = re.split(r"[\s,;]+", line.strip())
                if len(parts) >= 2:
                    try:
                        peaks.append((float(parts[0]), float(parts[1])))
                    except ValueError:
                        pass
            elif ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip()] = val.strip()

        if meta:
            yield {"meta": meta, "peaks": peaks}


In [3]:
# check instrument and instrument type to exclude specific given instruments
def check_instruments(instrument, instrument_type, excluded_df):
    if instrument in excluded_df["instrument"] or instrument_type in excluded_df["instrument_type"]:
        return False
    else:
        return True

In [4]:
def calculate_precursor(exactmass, adduct, mode):
    # so far only for [M+X]+ or [M-X]-
    splitter = "M+" if mode == "P" else "M-"
    mol = adduct.split(splitter)
    if len(mol) == 2:
        mol = mol[1].split("]")[0]
        if mode == "P":
            precursor_mz = float(exactmass) + Formula(mol).mass
        else:
            precursor_mz = float(exactmass) - Formula(mol).mass
        return precursor_mz
    return None

In [5]:
def calculate_exactmass(precursormz, adduct, mode):
    # so far only for [M+X]+ or [M-X]-
    splitter = "M+" if mode == "P" else "M-"
    mol = adduct.split(splitter)
    if len(mol) == 2:
        mol = mol[1].split("]")[0]
        if mode == "P":
            exactmass = float(precursormz) - Formula(mol).mass
        else:
            exactmass = float(precursormz) + Formula(mol).mass
        return exactmass
    return None

In [6]:
# split individual DB entries and only obtain needed information to create CompoundDB
#
# needed information: name, formula, exact mass, inchikey, splash, instrument, instrument type,
# polarity, precursor mz, mz, intensity, library source, collision energy, inchi and adduct
#
# DB entries without precursor mass are kept, this mass is calculated using the exact mass and adduct
#
# DB entries without peak information, polarity, without precursor mz and exact mass or that are not MS2 included in the combined DB
#
# if wanted, specific instrument types can be excluded. This option requires an additional csv file with the excluded names
# The file should contain one column with the excluded instruments and one with the excluded instrument types

def split_library_by_ionmode(msp_file, out_pos="gnps_pos.sqlite", out_neg="gnps_neg.sqlite",
                           batch_size=1000, DB="GNPS", check_instrument=False, ex_instruments=None):

    schema = """CREATE TABLE IF NOT EXISTS spectra (
        compound_id TEXT, name TEXT, inchikey TEXT, formula TEXT,
        adduct TEXT, precursor_mz REAL, polarity TEXT,
        collision_energy TEXT, mz TEXT, intensity TEXT, library_source TEXT,
        instrument_type TEXT, instrument TEXT, exactmass REAL,
        inchi TEXT, splash TEXT)"""
    insert_sql = """INSERT INTO spectra VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)"""

    con_pos, con_neg = sqlite3.connect(out_pos), sqlite3.connect(out_neg)
    con_pos.execute(schema); con_neg.execute(schema)

    batch_pos, batch_neg, n_pos, n_neg = [], [], 0, 0

    # open allowed instrument file, if it should be checked
    if check_instrument and ex_instruments != None:
        excluded_instruments = pd.read_csv(ex_instruments)
    else:
        check_instrument = False
        excluded_instruments = None

    for rec in parse_msp_stream(msp_file):


        if DB == "MoNA":

            # for MoNA we do not need to check the instruments at the moment, because we are only using the Orbitrap data
            meta, peaks = rec["meta"], rec["peaks"]
            if not peaks:
                continue

            # MoNA contains not only MS2 spectra, including them results in problems down the line and we are only interested in MS2
            if "PrecursorMZ" not in meta or meta["Spectrum_type"] != "MS2":
                continue

            mode_raw = meta.get("Ion_mode", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            # exclude, if instrument information is missing or if instrument is excluded
            if "Instrument_type" in meta and "Instrument" in meta and not meta["Instrument_type"] and not meta["Instrument"]:
                continue
            else:
                instrument = meta.get("Instrument")
                instrument_type = meta.get("Instrument_type")
                if check_instrument:
                    correct_instrument = check_instruments(instrument=instrument,
                                                           instrument_type=instrument_type,
                                                           excluded_df=excluded_instruments)
                    if not correct_instrument:
                        continue

            # exclude, if exact mass and precursor mz are missing. calculate if only one of them is missing
            if not meta["PrecursorMZ"]:
                if ("ExactMass" in meta and not meta["ExactMass"]) or ("Precursor_type" in meta and not meta["Precursor_type"]):
                    continue
                adduct = meta.get("Precursor_type")
                exactmass = meta.get("ExactMass")
                precursor_mz = calculate_precursor(exactmass = exactmass,
                                                   adduct = adduct,
                                                   mode = mode)
                if precursor_mz == None:
                    continue
            else:
                precursor_mz = float(meta.get("PrecursorMZ"))
                adduct = None if ("Precursor_type" not in meta or not meta["Precursor_type"]) else meta.get("Precursor_type")
                if ("ExactMass" in meta and not meta["ExactMass"]) and adduct is not None:
                    exactmass = calculate_exactmass(precursormz = precursor_mz,
                                                    adduct = adduct,
                                                    mode = mode)
                else:
                    exactmass = None

            # for MoNA, the Inchi is included in the comment
            if "InChI" in meta.get("Comments"):
                inchi = [i for i in meta.get("Comments").split('"') if "InChI" in i][0][6:]
            else:
                inchi = None

            # for MoNA, the splash is included in the comment
            if "SPLASH" in meta.get("Comments"):
                splash = [i for i in meta.get("Comments").split('"') if "SPLASH" in i][0][7:]
            else:
                splash = None

            row = (
                meta.get("DB#"), meta.get("Name"), meta.get("InChIKey"), meta.get("Formula"),
                meta.get("Precursor_type"),
                precursor_mz,
                mode, meta.get("Collision_energy"),
                ";".join(str(p[0]) for p in peaks), ";".join(str(p[1]) for p in peaks),
                "MoNA", meta.get("Instrument_type"), meta.get("Instrument"),
                exactmass, # either from library or calculated
                inchi, splash
            )

            if mode == "P":
                batch_pos.append(row); n_pos += 1
            else:
                batch_neg.append(row); n_neg += 1

        elif DB == "GNPS":

            # gnps does not seem to include the exact mass
            meta, peaks = rec["meta"], rec["peaks"]

            # exclude, if no peak information available
            if not peaks:
                continue

            # exclude, if polarity is missing
            mode_raw = meta.get("IONMODE", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            # exclude, if instrument information is missing or if instrument is excluded
            if not meta["INSTRUMENTTYPE"] and not meta["INSTRUMENT"]:
                continue
            else:
                instrument = meta.get("INSTRUMENT")
                instrument_type = meta.get("INSTRUMENTTYPE")
                if check_instrument:
                    correct_instrument = check_instruments(instrument=instrument,
                                                           instrument_type=instrument_type,
                                                           excluded_df=excluded_instruments)
                    if not correct_instrument:
                        continue

            # if mslevel is given and its not MS2, exclude
            if "MSLEVEL" in meta:
                print(meta.get("MSLEVEL"))

            # exclude, if exact mass and precursor mz are missing. calculate if only one of them is missing
            if not meta["PRECURSORMZ"]:
               continue
            else:
                precursor_mz = float(meta.get("PRECURSORMZ"))
                if precursor_mz == None:
                    continue
                adduct = meta.get("PRECURSORTYPE") if meta["PRECURSORTYPE"] else None
                exactmass = None


            meta["DB#"] = meta["Comment"].split(";")[0].split("=")[1]
            row = (
                meta.get("DB#"), meta.get("NAME"), meta.get("INCHIKEY"), meta.get("FORMULA"),
                adduct,
                precursor_mz, # either from library or calculated
                mode, meta.get("COLLISIONENERGY"),
                ";".join(str(p[0]) for p in peaks), #mz
                ";".join(str(p[1]) for p in peaks), #intensity
                "GNPS", instrument_type, instrument,
                exactmass, # either from library or calculated
                meta.get("INCHI"), None # no splash in DB
            )

            if mode == "P":
                batch_pos.append(row); n_pos += 1
            else:
                batch_neg.append(row); n_neg += 1

        elif DB == "MassBank":
            meta, peaks = rec["meta"], rec["peaks"]

            if not peaks:
                continue
           # MoNA contains not only MS2 spectra, including them results in problems down the line and we are only interested in MS2
            if "PrecursorMZ" not in meta or meta["Spectrum_type"] != "MS2":
                continue

            mode_raw = meta.get("Ion_mode", meta.get("Ionization mode", "")).upper()
            mode = "P" if mode_raw.startswith("P") else ("N" if mode_raw.startswith("N") else None)
            if mode is None:
                continue

            # exclude, if instrument information is missing or if instrument is excluded
            if "Instrument_type" in meta and "Instrument" in meta and not meta["Instrument_type"] and not meta["Instrument"]:
                continue
            else:
                instrument = meta.get("Instrument")
                instrument_type = meta.get("Instrument_type")
                if check_instrument:
                    correct_instrument = check_instruments(instrument=instrument,
                                                           instrument_type=instrument_type,
                                                           excluded_df=excluded_instruments)
                    if not correct_instrument:
                        continue

            # exclude, if exact mass and precursor mz are missing. calculate if only one of them is missing
            if not meta["PrecursorMZ"] or meta.get("PrecursorMZ") == "NA":
                if ("ExactMass" in meta and not meta["ExactMass"]) or ("Precursor_type" in meta and not meta["Precursor_type"]):
                    continue
                adduct = meta.get("Precursor_type")
                exactmass = meta.get("ExactMass")
                precursor_mz = calculate_precursor(exactmass = exactmass,
                                                   adduct = adduct,
                                                   mode = mode)
                if precursor_mz == None:
                    continue
            else:
                precursor_mz = float(meta.get("PrecursorMZ"))
                adduct = None if ("Precursor_type" not in meta or not meta["Precursor_type"]) else meta.get("Precursor_type")
                if ("ExactMass" in meta and not meta["ExactMass"]) and adduct is not None:
                    exactmass = calculate_exactmass(precursormz = precursor_mz,
                                                    adduct = adduct,
                                                    mode = mode)
                else:
                    exactmass = None


            row = (
                meta.get("DB#"), meta.get("Name"), meta.get("InChIKey"), meta.get("Formula"),
                adduct,
                precursor_mz, # either from DB or calculated
                mode, meta.get("Collision_energy"),
                ";".join(str(p[0]) for p in peaks),
                ";".join(str(p[1]) for p in peaks),
                "MassBank", instrument_type, instrument,
                exactmass, # either from DB or calculated
                meta.get("InChI"), meta.get("Splash")
            )

            if mode == "P":
                batch_pos.append(row); n_pos += 1
            else:
                batch_neg.append(row); n_neg += 1


        if len(batch_pos) >= batch_size:
            con_pos.executemany(insert_sql, batch_pos); con_pos.commit(); batch_pos = []
        if len(batch_neg) >= batch_size:
            con_neg.executemany(insert_sql, batch_neg); con_neg.commit(); batch_neg = []

    if batch_pos: con_pos.executemany(insert_sql, batch_pos); con_pos.commit()
    if batch_neg: con_neg.executemany(insert_sql, batch_neg); con_neg.commit()
    con_pos.close(); con_neg.close()
    print(f"Done. Positive: {n_pos}, Negative: {n_neg}")

In [7]:
split_library_by_ionmode(msp_file = "/home/walesch/Downloads/ALL_GNPS_NO_PROPOGATED.msp", out_pos="/home/walesch/PycharmProjects/metabolite_library_creation/gnps_pos.sqlite", out_neg="gnps_neg.sqlite", batch_size=1000, DB="GNPS", check_instrument=True, ex_instruments="/home/walesch/PycharmProjects/metabolite_library_creation/additional_information/excluded_instruments_gnps.csv")

Done. Positive: 713370, Negative: 240680


In [8]:
split_library_by_ionmode(msp_file= "/home/walesch/Downloads/MoNA-export-All_LC-MS-MS_Orbitrap-msp/MoNA-export-All_LC-MS-MS_Orbitrap.msp", out_pos = "/home/walesch/PycharmProjects/metabolite_library_creation/mona_pos.sqlite", out_neg = "mona_neg.sqlite", batch_size=1000, DB="MoNA")

Done. Positive: 99974, Negative: 57426


In [9]:
split_library_by_ionmode(msp_file = "/home/walesch/Downloads/MassBank_NISTformat.msp", out_pos = "massbank_pos.sqlite", out_neg = "/home/walesch/PycharmProjects/metabolite_library_creation/massbank_neg.sqlite", batch_size=1000, DB="MassBank", check_instrument=True, ex_instruments="/home/walesch/PycharmProjects/metabolite_library_creation/additional_information/excluded_instruments_massbank.csv")

Done. Positive: 75760, Negative: 37337
